In [ ]:
#AUXILARY FUNCTIONS
from PIL import Image
import numpy as np

def load_png(filename):
    # Open the PNG image file
    with Image.open(filename) as img:
        img = img.convert('RGBA')  # Ensure it has alpha channel
        # Extract info
        num_layers = 1  # PNGs are typically considered as a single-layer image
        num_channels = img.mode.count  # Count the number of channels based on the mode
        #get the dimensionality of the bits per channel through calculating the number of bits per channel
        bits_per_channel = 8
        # Convert image data to a numpy array
        array_n_channels = np.array(img)
    return (num_layers, num_channels, bits_per_channel, array_n_channels)

def load_bmp(filename):
    # Open the BMP image file
    with Image.open(filename) as img:
        img = img.convert('RGB')  # Ensure it is in RGB format
        # Extract info
        num_layers = 1  # BMPs are typically considered as a single-layer image
        num_channels = img.mode.count  # Count the number of channels based on the mode
        bits_per_channel = img.bits
        # Convert image data to a numpy array
        array_n_channels = np.array(img)
    return (num_layers, num_channels, bits_per_channel, array_n_channels)

from PIL import Image
import numpy as np

def export_rgba_png(filename, data, width, height):
    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, 4))
    elif data.ndim != 3 or data.shape != (height, width, 4):
        raise ValueError("Data must be a flat array or a 3D array with shape (height, width, 4).")
    
    # Create and save the image as PNG
    image = Image.fromarray(data, 'RGBA')
    image.save(filename, 'PNG', compress_level=0)

def export_rgba_bmp(filename, data, width, height):
    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, 4))
    elif data.ndim != 3 or data.shape != (height, width, 4):
        raise ValueError("Data must be a flat array or a 3D array with shape (height, width, 4).")

    # Create and save the image as BMP
    image = Image.fromarray(data, 'RGBA')
    image.save(filename, 'BMP')

    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, -1))
    elif data.ndim != 3:
        raise ValueError("Data must be a flat array or a 3D array with appropriate channel information.")
    
    # Create an Image object from the numpy array
    # Assume that data shape includes a channel dimension, e.g., (H, W, C) where C can be 1 (L), 3 (RGB), or 4 (RGBA)
    if data.shape[2] == 1:
        mode = 'L'  # Grayscale
    elif data.shape[2] == 3:
        mode = 'RGB'
    elif data.shape[2] == 4:
        mode = 'RGBA'
    else:
        raise ValueError("Unsupported number of channels. Data must have 1, 3, or 4 channels.")

    # Create the image
    image = Image.fromarray(data.astype('uint8'), mode)

    # Save the image with no compression
    image.save(filename, 'PNG', compress_level=0)
import numpy as np
from pydub import AudioSegment
import pydub
# Function to load an MP3 file and convert it to a numpy array
def load_mp3_to_array(file_path):
    # Load the MP3 file
    audio = AudioSegment.from_mp3(file_path)
    
    # Convert to single-channel (mono) if not already
    if audio.channels > 1:
        audio = audio.set_channels(1)
    
    # Get the raw audio data as a byte string and convert to a numpy array
    samples = np.array(audio.get_array_of_samples())
    print(len(samples))
    return samples, audio.frame_rate
from PIL import Image
import numpy as np

MASK = 0b11111000

def analyze_data_loss(image_filename,MASK=0b11111000):
    # Open the image
    with Image.open(image_filename) as img:
        # Convert image to RGB if not already (to handle JPEG and grayscale BMP)
        img = img.convert('RGB')
        
        # Convert image data to a numpy array
        original_data = np.array(img)

    # Mask to zero out the last two bits (binary: 11111100)

    # Apply the mask
    modified_data = original_data & MASK

    # Compute the absolute difference
    difference = np.abs(original_data - modified_data)

    # Calculate average difference per pixel
    average_difference = np.mean(difference)

    # Return the average difference
    return average_difference

def mask_rgb_array(data,MASK=0b11111000):
    # Mask to zero out the last two bits (binary: 11111100)

    # Apply the mask to each R, G , B channel of each pixel
    modified_data = data & MASK
    
    return modified_data


# Example usage:
filename = 'exploring_mp3_solution/jg.png'
loss = analyze_data_loss(filename)
print(f"Average data loss per color channel per pixel: {loss}")

#truncate the last two bits of each color channel
num_layers, num_channels, bits_per_channel, array_n_channels = load_png(filename)

modified_data = mask_rgb_array(array_n_channels)

#show me the first pixel's data before and after truncation
print(array_n_channels[0][0])
print(modified_data[0][0])

export_rgba_png('jg_truncated.png', modified_data, array_n_channels.shape[1], array_n_channels.shape[0])

#load the truncated image, and subtract the truncated from the original's data
filename = 'jg_truncated.png'
truncated_data = load_png(filename)[3]

#find the highest loss
loss = np.max(np.abs(array_n_channels - truncated_data))
print(f"Highest data loss per color channel per pixel: {loss}")

## Unified Testing Algorithm ##

In [ ]:
from PIL import Image
import numpy as np
from functools import partial

fill_empty_space_after_data_exhaustion = True

def encode_alpha(value):
    return value

def decode_alpha(value):
    return value

transformation_list = {3:(encode_alpha,decode_alpha)} # a list of tuples (<channel_index>,dynamic_encoding_function,dynamic_decoding_function)

def prepare_mask_operations(global_masking_operation):
    """
    Prepare mask operations to optimize computation in pixel manipulation functions.
    """
    shifts = {'R': 0, 'G': 1, 'B': 2, 'A': 3}
    mask_ops = {}
    for mask_value, shift_amount, channel in global_masking_operation:
        channel_index = shifts[channel]
        mask_ops[channel_index] = (mask_value, shift_amount)
    return mask_ops

def mask_and_embed(pixel, value, mask_ops,transformations = transformation_list):
    """
    Apply a mask to a pixel and embed a value into it using pre-calculated mask operations.
    """
    result_pixel = list(pixel)
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        # Extract appropriate bits from 'value'
        bits_to_embed = (value >> shift_amount) & mask_value
        # Mask out the bits in the original pixel and embed the new bits
        if channel_index in transformations:
            bits_to_embed = transformations[channel_index][0](bits_to_embed)
            result_pixel[channel_index] = (pixel[channel_index] & ~mask_value) | bits_to_embed
        else:
            result_pixel[channel_index] = (pixel[channel_index] & ~mask_value) | bits_to_embed
    return tuple(result_pixel)

def extract_from_mask(pixel, mask_ops,transformations=transformation_list):
    """
    Extract data from a pixel using pre-calculated mask operations.
    """
    extracted_value = 0
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        # Extract the bits from the pixel and position them correctly in the output value
        bits = (pixel[channel_index] & mask_value) << shift_amount
        if channel_index in transformations:
            bits = transformations[channel_index][1](bits)
        extracted_value |= bits
    return extracted_value

def unified_algorithm_v1(operation, image_filename, mask_scheme, endian='le', output_filename=None, yield_function=None, write_function=None,transformation_functions=transformation_list,fill_empty_space_after_data_exhaustion=True):
    """
    A universal function to handle both embedding (baking) and extracting (debaking) data in/from an image.
    """
    # Load image
    with Image.open(image_filename) as img:
        img_data = np.array(img)

    # Prepare mask operations
    mask_ops = prepare_mask_operations(mask_scheme)
    #for each channel, generate an empty transformation function

    if operation == 'bake':
        if yield_function is None:
            raise ValueError("yield_function must be provided for baking.")
        #if no alpha channel is present, add it
        if img_data.shape[2] == 3:
            img_data = np.dstack((img_data, np.full_like(img_data[:,:,0], 255)))
            print("Alpha channel added to the image data.")
        print("Shape of picture: ", img_data.shape)
        i, j = 0, 0
        for data in yield_function():
            if i >= img_data.shape[0]:
                break  # Stop if we run out of image space
            masked_value = mask_and_embed(img_data[i][j], data, mask_ops,transformation_functions)
            img_data[i][j] = masked_value
            j += 1
            if j >= img_data.shape[1]:
                i += 1
                j = 0
        #if we have ran out of data, fill the rest of the image with 0s
        if fill_empty_space_after_data_exhaustion:
            print("Data exhausted. Filling the rest of the image with zeros.")
            while i < img_data.shape[0]:
                img_data[i][j] = (0, 0, 0, 0)
                j += 1
                if j >= img_data.shape[1]:
                    i += 1
                    j = 0

        # Save the modified image
        new_img = Image.fromarray(img_data)
        output_filename = output_filename if output_filename else "output_image.png"
        new_img.save(output_filename)
        print(f"Baked image saved as {output_filename}")

    elif operation == 'debake':
        extracted_data = (extract_from_mask(pixel, mask_ops,transformation_functions) for row in img_data for pixel in row)

        if write_function is None:
            raise ValueError("write_function must be provided for debaking.")

        # Use write_function to handle the output of extracted data
        write_function(extracted_data)
        print(f"Extracted data written using the provided write function.")
    else:
        raise ValueError("Unsupported operation specified")


def test_masking_operation(value_to_embed = 0b11011010,global_masking_operation = [(0b00000011, 0, 'R'), (0b00000111, 2, 'G'), (0b00000111, 5, 'B'), (0b11111111, 8, 'A')]):
        
    print(f"Value to embed (bin): 0b{value_to_embed:08b}, (dec): {value_to_embed}")
    mask_ops = prepare_mask_operations(global_masking_operation)
    test_pixel = (100, 150, 200, 250)
    embedded_pixel = mask_and_embed(test_pixel, value_to_embed, mask_ops)
    extracted_value = extract_from_mask(embedded_pixel, mask_ops)

    print("Mask Operations:", mask_ops)
    print("Original Pixel:", test_pixel)
    print("Embedded Pixel:", embedded_pixel)
    print("Extracted Value: 0b{:08b}".format(extracted_value))
    print("Extracted value decimal: ", extracted_value)
    if extracted_value == value_to_embed:
        print("Test passed.")
    else:
        print("Test failed.")
        
test_masking_operation(value_to_embed=32123)

v1_masking = [
    (0b00000011, 0, 'R'), 
    (0b00000111, 2, 'G'), 
    (0b00000111, 5, 'B'),
    (0b11111111, 8, 'A')
]
test_masking_operation(global_masking_operation=v1_masking)

## TESTING ##


In [ ]:
import IPython.display as display
audio_file_to_encode = "audio_samples/4am_song.wav"

#v2 masking should start with A as we need the most significant bits to be the least affecting the image
# v2_masking = [
#     (0b11111111, 0, 'A'), 
#     (0b00000011, 8, 'R'), 
#     (0b00000111, 10, 'G'), 
#     (0b00000111, 13, 'B')
# ]
# # mustafa style
# v1_masking = [
#     (0b00000111, 0, 'G'),
#     (0b01111111, 3, 'A'),
#     (0b00000111, 10, 'R'),  
#     (0b00000111, 13, 'B')
# ]

v1_masking = [ #full balance
    (0b00001111, 0, 'R'), 
    (0b00001111, 4, 'G'), 
    (0b00001111, 8, 'B'),
    (0b00001111, 12, 'A')
]

# v2_masking = [ #base style, good, balanced
#     (0b01111111, 0, 'A'),
#     (0b00000111, 7, 'R'), 
#     (0b00000111, 10, 'G'), 
#     (0b00000111, 13, 'B')
# ]

v2_masking = [ #base style, good, balanced
    (0b01111111, 0, 'A'),
    (0b00000111, 7, 'R'),  
    (0b00000111, 10, 'B'),
    (0b00000111, 13, 'G')
]

# v2_masking = [ #base style, good, balanced
#     (0b00111111, 0, 'A'),
#     (0b00001111, 6, 'R'),  
#     (0b00000111, 10, 'B'),
#     (0b00000111, 13, 'G')
# ]

# v1_masking = [ #bakes the image too much...
#     (0b01111111, 0, 'A'),
#     (0b00010101, 7, 'R'), 
#     (0b00010101, 10, 'G'), 
#     (0b00010101, 13, 'B')
# ]


skip_initial_silence = True
def audio_read_yield_function():
    samples, frame_rate = load_mp3_to_array(audio_file_to_encode)
    if skip_initial_silence:
        for i, sample in enumerate(samples):
            if sample != 0:
                print(f"Skipping {i} samples of silence.")
                samples = samples[i:]
                break
    for sample in samples:
        yield sample

unified_algorithm_v1("bake","image_samples/jg.png", v1_masking,yield_function=audio_read_yield_function, output_filename="jg_baked.png")

#v1 masked vs v2 masked
unified_algorithm_v1("bake","image_samples/jg.png", v2_masking,yield_function=audio_read_yield_function, output_filename="jg_baked_v2.png")
display.display(display.Image(filename="jg_baked.png"))
display.display(display.Image(filename="jg_baked_v2.png"))


decoded_audio_output_filename = "extracted_audio_v2.wav"

def audio_write_function(data):
    audio = AudioSegment(
        data=np.array(list(data), dtype=np.int16).tobytes(),
        sample_width=2,
        frame_rate=48000,
        channels=1  # Mono
    )
    audio.export(decoded_audio_output_filename, format="wav")

unified_algorithm_v1("debake","jg_baked.png", v1_masking,write_function=audio_write_function)
#debake the v2 masked image
unified_algorithm_v1("debake","jg_baked_v2.png", v2_masking,write_function=audio_write_function)
print("V1 masking scheme")
display.display(display.Audio(decoded_audio_output_filename))
print("V2 masking scheme")
display.display(display.Audio(decoded_audio_output_filename))


In [ ]:
unified_algorithm_v1("bake","image_samples/8k_test_sample.png", v2_masking,yield_function=audio_read_yield_function, output_filename="8k_encoded_aria.png")

In [ ]:
#failed attempt to debake a screenshot of the encoded audio
# unified_algorithm_v1("debake","screenshot.png", v2_masking,write_function=audio_write_function)
# display.display(display.Audio(decoded_audio_output_filename))


In [ ]:
unified_algorithm_v1("debake","8k_encoded_aria.png", v2_masking,write_function=audio_write_function)
display.display(display.Audio(decoded_audio_output_filename))


In [ ]:
#laod the alpha channel of the v2 image, i need to know the distribution of the values in the alpha channel
#show me the distributions of each channel
def graph_channel_distribution(image_filepath):
    print("Channel distributions for the image: ", image_filepath)    
    num_layers, num_channels, bits_per_channel, array_n_channels = load_png(image_filepath)
    import matplotlib.pyplot as plt
    for i in range(4):
        #plot the distributions of each channel
        plt.hist(array_n_channels[:,:,i].flatten(), bins=256, range=(0, 256), color='c', edgecolor='k')
        plt.title(f"Channel {i}")
        plt.show()
def compare_channel_distributions(image_filepath1,image_filepath2):
    print("Channel distributions for the image: ", image_filepath1)    
    num_layers, num_channels, bits_per_channel, array_n_channels = load_png(image_filepath1)
    print("Channel distributions for the image: ", image_filepath2)    
    num_layers, num_channels, bits_per_channel, array_n_channels = load_png(image_filepath2)
    import matplotlib.pyplot as plt
    for i in range(4):
        #overlay the distributions of each channel
        plt.hist(array_n_channels[:,:,i].flatten(), bins=256, range=(0, 256), color='c', edgecolor='k', alpha=0.5)
        plt.title(f"Channel {i}")
        plt.show()

In [ ]:
compare_channel_distributions("jg_baked.png","jg_baked_v2.png")
# graph_channel_distribution("jg_baked.png")
# graph_channel_distribution("jg_baked_v2.png")


## GPU COMPUTE ##

In [ ]:
import cupy as cp
from PIL import Image
import numpy as np

def prepare_mask_operations_gpu(global_masking_operation):
    """
    Prepare mask operations to optimize computation in pixel manipulation functions for GPU.
    """
    shifts = {'R': 0, 'G': 1, 'B': 2, 'A': 3}
    mask_values = cp.zeros(4, dtype=cp.int32)
    shift_values = cp.zeros(4, dtype=cp.int32)
    for mask_value, shift_amount, channel in global_masking_operation:
        channel_index = shifts[channel]
        mask_values[channel_index] = mask_value
        shift_values[channel_index] = shift_amount
    return mask_values, shift_values

# Define the CuPy kernels for mask_and_embed and extract_from_mask
mask_and_embed_kernel = cp.ElementwiseKernel(
    'raw T img, raw S value, raw W mask, raw U shift', 'T out',
    '''
    int channel_index = i % 4;  // Assuming img is in RGBA format
    int mask_value = mask[channel_index];
    int shift_amount = shift[channel_index];

    int bits_to_embed = (value >> shift_amount) & mask_value;
    out = (img & ~mask_value) | bits_to_embed;
    ''',
    'mask_and_embed'
)

extract_from_mask_kernel = cp.ElementwiseKernel(
    'raw T img, raw W mask, raw U shift', 'T out',
    '''
    int extracted_value = 0;
    int channel_index = i % 4;  // Assuming img is in RGBA format
    int mask_value = mask[channel_index];
    int shift_amount = shift[channel_index];

    int bits = (img & mask_value) << shift_amount;
    extracted_value |= bits;
    out = extracted_value;
    ''',
    'extract_from_mask'
)

def unified_gpu_algorithm(operation, image_filename, mask_scheme, endian='le', output_filename=None, yield_function=None, write_function=None, transformation_list=None, fill_empty_space_after_data_exhaustion=True):
    """
    A universal function to handle both embedding (baking) and extracting (debaking) data in/from an image using GPU.
    """
    # Load image and convert to CuPy array
    with Image.open(image_filename) as img:
        img_data = cp.asarray(np.array(img))

    # Prepare mask operations for GPU
    mask_values, shift_values = prepare_mask_operations_gpu(mask_scheme)

    if operation == 'bake':
        if yield_function is None:
            raise ValueError("yield_function must be provided for baking.")

        if img_data.shape[2] == 3:
            alpha_channel = cp.full_like(img_data[:,:,0], 255)
            img_data = cp.dstack((img_data, alpha_channel))
            print("Alpha channel added to the image data.")
        print("Shape of picture: ", img_data.shape)

        i, j = 0, 0
        for data in yield_function():
            if i >= img_data.shape[0]:
                break
            mask_and_embed_kernel(img_data[i, j], data, mask_values, shift_values, img_data[i, j])
            j += 1
            if j >= img_data.shape[1]:
                i += 1
                j = 0

        if fill_empty_space_after_data_exhaustion:
            while i < img_data.shape[0]:
                img_data[i, j] = (0, 0, 0, 0)
                j += 1
                if j >= img_data.shape[1]:
                    i += 1
                    j = 0

        new_img = Image.fromarray(cp.asnumpy(img_data))
        output_filename = output_filename if output_filename else "output_image.png"
        new_img.save(output_filename)
        print(f"Baked image saved as {output_filename}")

    elif operation == 'debake':
        extracted_data = []
        for row in img_data:
            for pixel in row:
                extracted_data.append(extract_from_mask_kernel(pixel, mask_values, shift_values))

        if write_function is None:
            raise ValueError("write_function must be provided for debaking.")

        write_function(extracted_data)
        print(f"Extracted data written using the provided write function.")
    else:
        raise ValueError("Unsupported operation specified")



## Testing V6  COMPRESSION AND CESAR CYPHER OF BITS##

In [ ]:
v3_masking = [
    (0b11111111, 0, 'A'), 
    (0b00000011, 8, 'R'), 
    (0b00000111, 10, 'G'), 
    (0b00000111, 13, 'B')
]

def audio_read_yield_function_v3():
    samples, frame_rate = load_mp3_to_array(audio_file_to_encode) # loads wav too
    if skip_initial_silence:
        for i, sample in enumerate(samples):
            if sample != 0:
                print(f"Skipping {i} samples of silence.")
                samples = samples[i:]
                break
    for sample in samples:
        yield sample


#clear the previous functions from memory
if 'mask_and_embed' in globals():
    print("Deleting mask_and_embed")
    del mask_and_embed
if 'extract_from_mask' in globals():
    print("Deleting extract_from_mask")
    del extract_from_mask

#use the same algorithm as before, but this time, i need the data that is stored in the alpha channel to be sshifted by 10, so that the 0 values become 246 and the 255 values become 245
ceaser_cipher_shift = 0
def mask_and_embed(pixel, value, mask_ops):
    """
    Apply a mask to a pixel and embed a value into it using pre-calculated mask operations.
    """
    result_pixel = list(pixel)
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        # Extract appropriate bits from 'value'
        bits_to_embed = (value >> shift_amount) & mask_value
        #rotate the values in the alpha channel like a ceaser cipher
        if channel_index == 3:
            bits_to_embed = (bits_to_embed + ceaser_cipher_shift) % 256
        result_pixel[channel_index] = (pixel[channel_index] & ~mask_value) | bits_to_embed
    return tuple(result_pixel)
def extract_from_mask(pixel, mask_ops):
    """
    Extract data from a pixel using pre-calculated mask operations.
    """
    extracted_value = 0
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        if channel_index == 3:
            shift_amount = (shift_amount + ceaser_cipher_shift) % 256
        bits = (pixel[channel_index] & mask_value) << shift_amount
        extracted_value |= bits
    return extracted_value

#pass the new masking scheme to the algorithm
unified_algorithm_v1("bake","image_samples/jg.png",v3_masking,yield_function=audio_read_yield_function, output_filename="jg_baked_v6.png")
display.display(display.Image(filename="jg_baked_v6.png"))
#debake the v3 masked image
unified_algorithm_v1("debake","jg_baked_v6.png", v3_masking,write_function=audio_write_function)
display.display(display.Audio(decoded_audio_output_filename))
#show the distributions before and after the bit rotation
graph_channel_distribution("jg_baked_v6.png")

## Compressing and decompressing the audio file, and just writing bytes, need to implement size checking if the compressed file would fit the space ##

In [ ]:
import gzip
#calculate the number of bits that we can store in the image
def calculate_capacity(image_filename, mask_scheme):
    # Load image
    with Image.open(image_filename) as img:
        img_data = np.array(img)
    # Prepare mask operations
    mask_ops = prepare_mask_operations(mask_scheme)
    # Calculate the number of bits that can be stored in the image
    bits_per_pixel = sum(mask[0].bit_length() for mask in mask_scheme)
    total_bits = img_data.size * bits_per_pixel
    return total_bits

def compress_audio_file(audio_file):
    # Compress the audio file using the deflate algorithm
    compressed_path = "compressed_audio.gz"
    with open(audio_file, 'rb') as f_in, gzip.open(compressed_path, 'wb') as f_out:
        f_out.writelines(f_in)  # Stream data into gzip file
    return compressed_path

def audio_read_yield_function_v4():
    compressed_path = compress_audio_file(audio_file_to_encode)
    with open(compressed_path, 'rb') as f:
        compressed_data = f.read()
    print(f"Compressed audio size: {len(compressed_data)} bytes")
    for byte in compressed_data:
        yield byte

def decompress_audio_file(compressed_path, output_file):
    # Decompress the given file
    with gzip.open(compressed_path, 'rb') as f_in, open(output_file, 'wb') as f_out:
        f_out.writelines(f_in)

def write_function_for_unified_algorithm_v4(data):
    compressed_path = "extracted_file.gz"
    # Append data to the compressed file
    with open(compressed_path, 'ab') as f:
        f.write(bytes([data]))
    # Decompress the file to extract the audio
    decompress_audio_file(compressed_path, "extracted_audio_v2_decompressed.wav")

def mask_and_embed(pixel, value, mask_ops):
    """
    Apply a mask to a pixel and embed a value into it using pre-calculated mask operations.
    """
    result_pixel = list(pixel)
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        # Extract appropriate bits from 'value'
        bits_to_embed = (value >> shift_amount) & mask_value
        # Mask out the bits in the original pixel and embed the new bits
        result_pixel[channel_index] = (pixel[channel_index] & ~mask_value) | bits_to_embed
    return tuple(result_pixel)

def extract_from_mask(pixel, mask_ops):
    """
    Extract data from a pixel using pre-calculated mask operations.
    """
    extracted_value = 0
    for channel_index, (mask_value, shift_amount) in mask_ops.items():
        # Extract the bits from the pixel and position them correctly in the output value
        bits = (pixel[channel_index] & mask_value) << shift_amount
        extracted_value |= bits
    return extracted_value

# file_to_encode = "audio_samples/anguish.wav"
# image_to_bake = "image_samples/jg.png"
# print(f"Capacity of the image using v3 masking scheme: {calculate_capacity(image_to_bake, v3_masking)/8}")

# import os
# file_size = os.path.getsize(file_to_encode)
# print(f"Size of the file to encode: {file_size} bytes")
# if file_size > calculate_capacity(image_to_bake, v3_masking)/8:
#     print("File too big to store in the image.")
#     if input("Do you want to continue? (y/n): ").lower() != 'y':
#         raise ValueError("Operation cancelled by user.")

# unified_algorithm_v1("bake",image_to_bake,v3_masking,yield_function=audio_read_yield_function_v4, output_filename="jg_baked_v7.png")
# display.display(display.Image(filename="jg_baked_v7.png"))


# unified_algorithm_v1("debake","jg_baked_v7.png", v3_masking,write_function=write_function_for_unified_algorithm_v4)
# display.display(display.Audio("extracted_audio_v2_decompressed.wav"))

Direct Byte Insertion

In [ ]:
v5_testing_masking = [
    (0b11111111, 0, 'A'),
    (0b11111111, 8, 'R'),
    (0b11111111, 16, 'G'),
    (0b11111111, 24, 'B')
]

file_to_encode = "audio_samples/anguish.wav"
image_to_bake = "image_samples/jg.png"
file_to_write = "extracted_file_v5.wav"

def byte_yield_function():
    #read all the bytes, then return them 16 bits at a time    
    with open(file_to_encode, 'rb') as f:
        data = f.read()
    
    if skip_initial_silence:
        for i, byte in enumerate(data):
            if byte != 0:
                print(f"Skipping {i} bytes of silence.")
                data = data[i:]
                break
            
    for i in range(0, len(data), 4):
        #return a 24 bit byte
        yield int.from_bytes(data[i:i+4], 'little')

def byte_write_function(data_generator):
    if os.path.exists(file_to_write):
        os.remove(file_to_write)
    # Open the file to write the extracted bytes
    with open(file_to_write, 'ab') as f:
        # Iterate through the generator
        for data in data_generator:
            # Assuming data is in a correct byte format or needs to be converted from an integer
            #save it as many bytes as it needs
            byte_data = int(data).to_bytes(4, 'little')
            f.write(byte_data)

import os
file_size = os.path.getsize(file_to_encode)
print(f"Size of the file to encode: {file_size} bytes")
if file_size > calculate_capacity(image_to_bake, v5_testing_masking)/8:
    print("File too big to store in the image.")
    if input("Do you want to continue? (y/n): ").lower() != 'y':
        raise ValueError("Operation cancelled by user.")

unified_algorithm_v1("bake",image_to_bake,v5_testing_masking,yield_function=byte_yield_function, output_filename="jg_baked_v7.png")
display.display(display.Image(filename="jg_baked_v7.png"))

unified_algorithm_v1("debake","jg_baked_v7.png", v5_testing_masking,write_function=byte_write_function)

#diff the original and the extracted files to see if they are the same, byte by byte

In [ ]:
#go through each byte of the 2 files and compare them
def diff_files(file1, file2):
    with open(file1, 'rb') as f1, open(file2, 'rb') as f2:
        for i, (byte1, byte2) in enumerate(zip(f1, f2)):
            if byte1 != byte2:
                print(f"Files differ at byte {i}: \n{byte1}\nvs\n{byte2}")
                return
    print("Files are identical.")
    #go through each byte of the 2 files and compare them

diff_files("extracted_audio_v2.wav","extracted_file_v5.wav")